In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import scvi
import torch
import warnings
import leidenalg as la
import anndata
from matplotlib import pyplot as plt
from matplotlib.pyplot import rc_context
from scvi.autotune import ModelTuner
from ray import tune
import glob
import re
import csv
import itertools
from scipy.io import mmwrite
import pynndescent
import umap
import random
warnings.filterwarnings('ignore')
sc.set_figure_params(dpi=200)
plt.rcParams['figure.figsize'] = [3,3]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device
from pathlib import Path
def _resolve_partition_root():
    cwd = Path(os.getcwd()).resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "primary_script").exists() and (p / "intermediate").exists():
            return p
        candidate = p / "technical_review" / "submission_partition_draft"
        if (candidate / "primary_script").exists():
            return candidate
    raise RuntimeError("Could not locate submission_partition_draft root")
REPO_ROOT = _resolve_partition_root()


In [ ]:
os.chdir(str(REPO_ROOT / 'intermediate'))

In [ ]:
adata_myeloid = sc.read_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_myeloid.h5ad'))
adata_platelet = sc.read_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_subset_platelet.h5ad'))

In [ ]:
adata_myeloid.obs['merged_type'].value_counts()

In [ ]:
unique_pid = adata_myeloid.obs['study_id'].values.unique(); unique_pid

In [ ]:
def generate_oligo_tag(length=16):
    return ''.join(random.choices('CGAT', k=length))

def generate_unique_oligo_tags(num_tags, length=16):
    unique_tags = set()
    while len(unique_tags) < num_tags:
        new_tag = generate_oligo_tag(length) + '-dbl'
        unique_tags.add(new_tag)
    return list(unique_tags)

In [ ]:
adata_cmo = adata_myeloid[adata_myeloid.obs['merged_type']=='CD14 Mono']
adata_plt = adata_platelet[adata_platelet.obs['merged_type']=='Platelet'] # doesn't actually drop any cells, but just to be safe

In [ ]:
common_genes = np.intersect1d(adata_cmo.var_names, adata_plt.var_names)

adata_cmo = adata_cmo[:, common_genes]
adata_plt = adata_plt[:, common_genes]

In [ ]:
assert np.all(adata_cmo.layers['counts'].data == adata_cmo.layers['counts'].data.astype(np.int32))
assert np.all(adata_plt.layers['counts'].data == adata_plt.layers['counts'].data.astype(np.int32))
assert np.mean(adata_cmo.var_names==adata_plt.var_names)==1.0

In [ ]:
adata_cmo.obs['merged_type'].value_counts()

In [ ]:
# will create this many simulated MPA
adata_plt.obs['merged_type'].value_counts()

In [ ]:
adata_cmo_sd1 = adata_cmo[adata_cmo.obs['study_day']=='SD1']
adata_cmo_sd2 = adata_cmo[adata_cmo.obs['study_day']=='SD2']
adata_cmo_sd3 = adata_cmo[adata_cmo.obs['study_day']=='SD3']
adata_plt_sd1 = adata_plt[adata_plt.obs['study_day']=='SD1']
adata_plt_sd2 = adata_plt[adata_plt.obs['study_day']=='SD2']
adata_plt_sd3 = adata_plt[adata_plt.obs['study_day']=='SD3']

In [ ]:
doublet_list = []
np.random.seed(123)
for i in range(len(unique_pid)):
    tmp_id = unique_pid[i]
    print('formulating anndata object with simulated cMo + Platelet doublets for pid: [' + tmp_id + '] ', str(i+1), ' of ',len(unique_pid), ' pid ...')
    
    sd1_cmo = adata_cmo_sd1[adata_cmo_sd1.obs['study_id']==tmp_id]
    sd2_cmo = adata_cmo_sd2[adata_cmo_sd2.obs['study_id']==tmp_id]
    sd3_cmo = adata_cmo_sd3[adata_cmo_sd3.obs['study_id']==tmp_id]
    sd1_plt = adata_plt_sd1[adata_plt_sd1.obs['study_id']==tmp_id]
    sd2_plt = adata_plt_sd2[adata_plt_sd2.obs['study_id']==tmp_id]
    sd3_plt = adata_plt_sd3[adata_plt_sd3.obs['study_id']==tmp_id]
    # sd1
    min_sd1 = np.min([sd1_cmo.shape[0],sd1_plt.shape[0]])
    sd1_cmo = sd1_cmo[np.random.choice(sd1_cmo.n_obs, size=min_sd1, replace=False)]
    sd1_plt = sd1_plt[np.random.choice(sd1_plt.n_obs, size=min_sd1, replace=False)]
    #sd1_sim_ct = sd1_cmo.X + sd1_plt.X
    sd1_sim_ct = sd1_cmo.layers['counts'] + sd1_plt.layers['counts']
    assert sd1_cmo.n_obs == sd1_plt.n_obs, "The number of cells in both [SD1] AnnData objects must be the same."
    assert all(sd1_cmo.var_names == sd1_plt.var_names), "The feature names (var_names) in both [SD1] AnnData objects must be the same."
    sd1_adata_sim = anndata.AnnData(sd1_sim_ct, var=sd1_cmo.var)
    sd1_adata_sim.obs['lane'] = [x if x==y else x+'_'+y for x,y in zip(sd1_cmo.obs['lane'],sd1_plt.obs['lane'])]
    sd1_adata_sim.obs['study_day'] = 'SD1'
    # sd2
    min_sd2 = np.min([sd2_cmo.shape[0],sd2_plt.shape[0]])
    sd2_cmo = sd2_cmo[np.random.choice(sd2_cmo.n_obs, size=min_sd2, replace=False)]
    sd2_plt = sd2_plt[np.random.choice(sd2_plt.n_obs, size=min_sd2, replace=False)]
    #sd2_sim_ct = sd2_cmo.X + sd2_plt.X
    sd2_sim_ct = sd2_cmo.layers['counts'] + sd2_plt.layers['counts']
    assert sd2_cmo.n_obs == sd2_plt.n_obs, "The number of cells in both [SD2] AnnData objects must be the same."
    assert all(sd2_cmo.var_names == sd2_plt.var_names), "The feature names (var_names) in both [SD2] AnnData objects must be the same."
    sd2_adata_sim = anndata.AnnData(sd2_sim_ct, var=sd2_cmo.var)
    sd2_adata_sim.obs['lane'] = [x if x==y else x+'_'+y for x,y in zip(sd2_cmo.obs['lane'],sd2_plt.obs['lane'],)]
    sd2_adata_sim.obs['study_day'] = 'SD2'
    # sd3
    min_sd3 = np.min([sd3_cmo.shape[0],sd3_plt.shape[0]])
    sd3_cmo = sd3_cmo[np.random.choice(sd3_cmo.n_obs, size=min_sd3, replace=False)]
    sd3_plt = sd3_plt[np.random.choice(sd3_plt.n_obs, size=min_sd3, replace=False)]
    #sd3_sim_ct = sd3_cmo.X + sd3_plt.X
    sd3_sim_ct = sd3_cmo.layers['counts'] + sd3_plt.layers['counts']
    assert sd3_cmo.n_obs == sd3_plt.n_obs, "The number of cells in both [SD3] AnnData objects must be the same."
    assert all(sd3_cmo.var_names == sd3_plt.var_names), "The feature names (var_names) in both [SD3] AnnData objects must be the same."
    sd3_adata_sim = anndata.AnnData(sd3_sim_ct, var=sd3_cmo.var)
    sd3_adata_sim.obs['lane'] = [x if x==y else x+'_'+y for x,y in zip(sd3_cmo.obs['lane'],sd3_plt.obs['lane'],)]
    sd3_adata_sim.obs['study_day'] = 'SD3'

    doublet_adata = anndata.concat([sd1_adata_sim, sd2_adata_sim, sd3_adata_sim])
    num_tags = doublet_adata.n_obs
    unique_oligo_tags = generate_unique_oligo_tags(num_tags)
    oligo_tags = [generate_oligo_tag() + '-dbl' for _ in range(doublet_adata.n_obs)]
    doublet_adata.obs_names = oligo_tags
    doublet_adata.obs['study_id'] = tmp_id
    doublet_adata.obs['map_sd'] = [x.replace('-','').upper()+'_'+y.replace('MM-','') for x,y in zip(doublet_adata.obs['study_id'],doublet_adata.obs['lane'])]
    doublet_adata.obs['souporcell_assignment'] = tmp_id
    doublet_adata.obs['barcode_2'] = doublet_adata.obs_names
    doublet_adata.obs['merged_type'] = 'cMo_Platelet_doublet'
    doublet_adata.var['MT'] = doublet_adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(doublet_adata, inplace=True, qc_vars = ['MT'])
    doublet_list.append(doublet_adata)
concatenated_doublets = anndata.concat(doublet_list)

In [ ]:
adata_myeloid.X = adata_myeloid.layers['counts']
adata_plt.X = adata_plt.layers['counts']

In [ ]:
sim_adata_concat = anndata.concat([adata_myeloid, adata_plt, concatenated_doublets])

In [ ]:
assert np.all(sim_adata_concat.X.data == sim_adata_concat.X.data.astype(np.int32))

In [ ]:
sim_adata_concat.obs['merged_type'].value_counts()

In [ ]:
sim_adata_concat.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_simulation_dataset.h5ad')) # cache write

In [ ]:
adata = sc.read_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_simulation_dataset.h5ad'))
sc.pp.filter_cells(adata, min_genes = 200)
sc.pp.filter_genes(adata, min_cells = 10)
adata.var['MT'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['MT'], percent_top=None, log1p=False, inplace=True)
adata = adata[adata.obs['pct_counts_MT'] <= 10]
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=2500, subset = False, layer = 'counts', 
                            flavor = "seurat_v3", batch_key="lane")
adata_var = adata.var
csv_data = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'EXCLUDE_XY_TCR_IG.csv'), header = None)

positions = np.where(adata.var_names.isin(csv_data[0]))[0]
adata.var.iloc[positions, adata.var.columns.get_loc('highly_variable')] = False
adata.var['highly_variable'].value_counts()

In [ ]:
adata.obs['merged_type'].value_counts()

In [ ]:
model_cls = scvi.model.SCVI
model_cls.setup_anndata(adata = adata, layer = "counts", batch_key = 'lane', 
                        categorical_covariate_keys=['study_id'], 
                        continuous_covariate_keys=['pct_counts_MT', 'total_counts'])
tuner = ModelTuner(model_cls)
search_space = {
    "n_hidden": tune.choice([92, 128, 192, 256]),
    "n_latent": tune.choice([10, 20, 30, 40, 50, 60]),
    "n_layers": tune.choice([1, 2, 3]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "gene_likelihood": tune.choice(["nb", "zinb"])}
results = tuner.fit(adata, metric="validation_loss",
                    resources = {'gpu': 1}, 
                    search_space = search_space,
                    num_samples = 100,
                    max_epochs = 20)

In [ ]:
import ray
ray.shutdown()

In [ ]:
import math
import re

def extract_from_path(path):
    m = re.search(
        r"gene_likelihood=([^,]+),lr=([^,]+),n_hidden=([^,]+),n_latent=([^,]+),n_layers=([^_]+)",
        path or ""
    )
    if not m:
        return {}
    return {
        "gene_likelihood": m.group(1),
        "lr": float(m.group(2)),
        "n_hidden": int(m.group(3)),
        "n_latent": int(m.group(4)),
        "n_layers": int(m.group(5)),
    }

records = []
for r in getattr(results, "results", []):
    loss = (getattr(r, "metrics", {}) or {}).get("validation_loss", math.inf)
    cfg = getattr(r, "config", None) or {}
    if not cfg:
        cfg = extract_from_path(getattr(r, "path", ""))

    records.append({
        "validation_loss": loss,
        "config": cfg,
        "path": getattr(r, "path", None),
        "result_obj": r,
    })

records = [x for x in records if math.isfinite(x["validation_loss"])]
records.sort(key=lambda x: x["validation_loss"])

best = records[0]
print("best validation_loss:", best["validation_loss"])
print("best config:", best["config"])
print("best path:", best["path"])

In [ ]:
# after tuning
top_param = {'n_hidden': 192, 
'n_latent': 60, 
'n_layers': 1, 
'gene_likelihood': 'zinb', 
'lr': 0.005632571262773357}

In [ ]:
scvi.model.SCVI.setup_anndata(adata = adata, layer = "counts", batch_key = 'lane', 
                              categorical_covariate_keys=['study_id'], 
                              continuous_covariate_keys=['pct_counts_MT', 'total_counts'])
model = scvi.model.SCVI(adata, n_hidden = top_param['n_hidden'], 
                        n_latent = top_param['n_latent'], 
                        n_layers = top_param['n_layers'], 
                        gene_likelihood = top_param['gene_likelihood'])
kwargs = {'lr': top_param['lr']}
model.train(max_epochs = 200, early_stopping = True, plan_kwargs = kwargs)

In [ ]:
y = model.history['reconstruction_loss_validation']['reconstruction_loss_validation'].min()
plt.plot(model.history['reconstruction_loss_train']['reconstruction_loss_train'], label='train')
plt.plot(model.history['reconstruction_loss_validation']['reconstruction_loss_validation'], label='validation')

plt.axhline(y, c = 'k')

plt.legend()
plt.show()

In [ ]:
# cache result; for reading on rerun
#adata.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'temp_pbmc_myeloid_platelet_sim.h5ad'))
adata = sc.read_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'temp_pbmc_myeloid_platelet_sim.h5ad'))
#model.save(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_myeloid_platelet_scvi_integration_model_sim'))
model = scvi.model.SCVI.load(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_myeloid_platelet_scvi_integration_model_sim/'), adata)

In [ ]:
adata.obsm['X_scVI'] = model.get_latent_representation()
scvi_norm_count = model.get_normalized_expression(library_size = 1e4)
adata.layers['scvi_normalized'] = scvi_norm_count

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI', random_state = 123)

In [ ]:
sc.tl.leiden(adata, resolution = 2, key_added = 'overcluster')

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.set_figure_params(dpi=300)
sc.pl.umap(adata, color = 'lane', size = 4, legend_fontsize = 'medium')

In [ ]:
sc.pl.umap(adata, color = 'study_id', size = 4, legend_fontsize = 'medium')

In [ ]:
sc.pl.umap(adata, color = 'study_day', size = 4, legend_fontsize = 'medium')

In [ ]:
sc.set_figure_params(dpi=200)
sc.pl.umap(adata, color = ['CD14','FCGR3A','LYZ','PLD4','ITM2C','S100A9','NEAT1','LILRA4','CLEC9A','PF4','FCER1A','IRF4'], size = 4, frameon = False, layer = 'scvi_normalized')

In [ ]:
sc.set_figure_params(dpi=100)
plt.rcParams['figure.figsize'] = [7,7]
sc.pl.umap(adata, color = ['overcluster'], legend_loc = 'on data', size = 3, legend_fontsize = 'x-small')

In [ ]:
sc.set_figure_params(dpi=100)
sc.pl.dotplot(adata, ['CD14','FCGR3A','LYZ','PLD4','ITM2C','S100A9','NEAT1','LILRA4','CLEC9A','IDO1','HLA-DQA1','FCER1A',
                      'CLEC10A','IRF4','MKI67','GNG11','PPBP','PF4','CAVIN2','TUBB1','CCR7'], groupby = 'overcluster', swap_axes = True,
              use_raw = True, standard_scale = 'var', dendrogram = False)

In [ ]:
# breakdown of top clusters by capture rate of previous platelet-derivatives
freq = pd.crosstab(adata.obs["merged_type"], adata.obs["overcluster"])
row_pct = freq.div(freq.sum(axis=1), axis=0).mul(100)
# optional: round for display
row_pct = row_pct.round(2)
print(row_pct.reindex(columns=['3', '13', '19'], fill_value=0))

In [ ]:
from IPython.display import display

rows = ["M-platelet", "cMo_Platelet_doublet", "Platelet"]
view = row_pct.loc[rows]
with pd.option_context("display.max_columns", None):
    display(view)

In [ ]:
#import matplotlib.pyplot as plt
import seaborn as sns

rows = ["M-platelet", "cMo_Platelet_doublet", "Platelet"]
view = row_pct.loc[rows]   # already in percent units

plt.figure(figsize=(16, 3.2))
ax = sns.heatmap(
    view,
    cmap="YlOrRd",
    vmin=0, vmax=100,
    annot=False, 
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Row-normalized %"}
)

ax.set_xlabel("overcluster")
ax.set_ylabel("previous labels")
ax.set_title("Merged Type vs Overcluster (Row-normalized %)")
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
sc.set_figure_params(dpi=100)
plt.rcParams['figure.figsize'] = [7,7]
# map previous singlet + simMPA labels
sc.pl.umap(adata, color = ['merged_type'], legend_loc = 'on data', size = 3, legend_fontsize = 'x-small')

In [ ]:
# initiate with singlet label
adata.obs['droplet_type'] = 'singlet'
adata.obs['droplet_type'] = ['MPA' if x=='M-platelet' else y for x,y in zip(adata.obs['merged_type'],adata.obs['droplet_type'])]
adata.obs['droplet_type'] = ['SimMPA' if x=='cMo_Platelet_doublet' else y for x,y in zip(adata.obs['merged_type'],adata.obs['droplet_type'])]
adata.obs['droplet_type'].value_counts()

In [ ]:
sc.pl.umap(adata, color = ['droplet_type'], size = 6, legend_fontsize = '8')

In [ ]:
adata.write_h5ad(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'pbmc_myeloid_platelet_int_sim.h5ad'))

In [ ]:
np.savetxt(fname = str(REPO_ROOT / 'intermediate' / 'pbmc' / 'anndata_elements/pbmc_mpa_sim_umap_coordinates.csv'), 
           X = adata.obsm['X_umap'], delimiter = ',')

In [ ]:
rna_counts = adata.layers['counts'] # whole integer counts, no normalization
adata_obs = adata.obs
adata_var = adata.var
mmwrite(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'anndata_elements/adata_pbmc_counts_mpa_sim.mtx'), rna_counts)
adata_obs.to_csv(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'anndata_elements/adata_pbmc_obs_mpa_sim.csv'))
adata_var.to_csv(str(REPO_ROOT / 'intermediate' / 'pbmc' / 'anndata_elements/adata_pbmc_var_mpa_sim.csv'))